# Execution Tracing

Agents can feel like black boxes — you know what went in and what came out, but not what happened in between. `AgentTrace` records the complete execution trajectory: every observation, every LLM decision, every tool call, organized by phases. This lets you replay, analyze, and debug agent behavior with full visibility.

In this tutorial, we'll run a **form-filling agent** in amphibious mode with tracing enabled, then inspect the trace to understand exactly how workflow failures triggered agent fallbacks and how the agent resolved them.

## Initialize

First, let's set up the LLM and define the tools our form-filling agent will use.

In [ ]:
import os
from bridgic.model import OpenAILlm

api_key = os.environ.get("OPENAI_API_KEY", "your-api-key")
base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

llm = LLM(model="openai/gpt-4o-mini")

In [ ]:
from bridgic.core import tool

state = {"captcha_shown": False, "form_submitted": False}

@tool(description="Fill a form field with a value")
async def fill_field(field_name: str, value: str) -> str:
    return f"Filled '{field_name}' with '{value}'"

@tool(description="Click a button on the page")
async def click_button(button_name: str) -> str:
    if button_name == "submit" and not state["captcha_shown"]:
        state["captcha_shown"] = True
        raise RuntimeError("CAPTCHA detected! Submit blocked.")
    state["form_submitted"] = True
    return f"Clicked '{button_name}' — success"

@tool(description="Solve the CAPTCHA challenge")
async def solve_captcha() -> str:
    return "CAPTCHA solved successfully"

@tool(description="Check current page status")
async def check_page() -> str:
    if state["captcha_shown"] and not state["form_submitted"]:
        return "Page shows: CAPTCHA challenge active, form fields are filled"
    return "Page shows: Form ready"

We have four tools:

- **`fill_field`** — fills a form field with a given value.
- **`click_button`** — clicks a button. On the first submit attempt, it simulates a CAPTCHA challenge by raising a `RuntimeError`.
- **`solve_captcha`** — solves the CAPTCHA challenge.
- **`check_page`** — reports the current state of the page.

The `state` dictionary simulates a real-world scenario: the first time we try to submit the form, a CAPTCHA appears. After solving it, subsequent submissions succeed.

## Enabling Execution Tracing

Enable tracing by passing `trace_running=True` to `arun()`. The framework will record every step of the execution — workflow steps, agent fallbacks, tool calls, and their results — into a structured `AgentTrace` object.

Let's build a form-filling agent in amphibious mode and run it with tracing enabled.

In [ ]:
from bridgic.amphibious import (
    AmphibiousAutoma, CognitiveContext, CognitiveWorker,
    think_unit, step, RunMode,
)

class TracedFormFiller(AmphibiousAutoma[CognitiveContext]):
    fixer = think_unit(
        CognitiveWorker.inline("Diagnose the problem and fix it so the form can be submitted."),
        max_attempts=5,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.fixer

    async def on_workflow(self, ctx: CognitiveContext):
        yield step("fill_field", field_name="username", value="alice")
        yield step("fill_field", field_name="email", value="alice@example.com")
        yield step("click_button", button_name="submit")

In [ ]:
# Reset state
state["captcha_shown"] = False
state["form_submitted"] = False

agent = TracedFormFiller(llm=llm)
result = await agent.arun(
    goal="Fill and submit the registration form",
    tools=[fill_field, click_button, solve_captcha, check_page],
    mode=RunMode.AMPHIBIOUS,
    will_fallback=True,
    trace_running=True,  # Enable tracing
)
print(result)

The agent ran in amphibious mode: the workflow filled two form fields successfully, then the submit step hit the CAPTCHA error. The framework automatically activated `on_agent()`, which diagnosed the problem, solved the CAPTCHA, and re-submitted the form.

All of this was recorded in the trace. Let's look inside.

## Accessing the Trace

After `arun()` completes, the trace is available on the agent. Call `build()` to get a structured dictionary representation of the entire execution trajectory.

In [ ]:
# Build the trace as a dictionary
trace = agent._agent_trace.build()

# Inspect the top-level structure
print("Trace keys:", list(trace.keys()))
print("Number of phases:", len(trace.get("phases", [])))
print("Orphan steps:", len(trace.get("orphan_steps", [])))

The trace dictionary has two main sections:

- **`phases`** — a list of execution phases (from Phase Annotations or auto-detected mode switches). Each phase contains its own steps.
- **`orphan_steps`** — steps that were not associated with any phase. In a well-structured agent, this should be empty or minimal.

## Inspecting Phases and Steps

Each phase in the trace contains detailed information about what happened during that part of the execution. Let's walk through each phase and its steps.

In [ ]:
# Walk through each phase
for i, phase in enumerate(trace.get("phases", [])):
    print(f"\n--- Phase {i+1} ---")
    print(f"  Type: {phase.get('phase_type', 'unknown')}")
    print(f"  Goal: {phase.get('goal', 'N/A')}")
    steps = phase.get("steps", [])
    print(f"  Steps: {len(steps)}")
    for j, s in enumerate(steps):
        print(f"    Step {j+1}: {s.get('step_content', '')[:80]}...")
        if s.get("tool_calls"):
            for tc in s["tool_calls"]:
                print(f"      Tool: {tc.get('tool_name', '?')}({tc.get('tool_arguments', {})})")

The trace reveals the full execution trajectory:

1. **Workflow phase** — the deterministic steps (`fill_field` twice, then the failed `click_button`).
2. **Agent phase** — the LLM-driven fallback that diagnosed the CAPTCHA, solved it, and re-submitted.

Each step records the content (what the LLM said or what the workflow step was), the tool calls made, and their results. This is the complete picture of what happened inside the black box.

## Saving and Loading Traces

Traces can be saved to JSON files for later analysis, sharing with teammates, or building dashboards. The `AgentTrace` class provides `save()` and `load()` methods for this.

In [ ]:
# Save trace to a JSON file
agent._agent_trace.save("form_filler_trace.json")
print("Trace saved to form_filler_trace.json")

# Load it back
from bridgic.amphibious import AgentTrace
loaded_trace = AgentTrace.load("form_filler_trace.json")
print("Loaded trace phases:", len(loaded_trace.get("phases", [])))

Saved traces are plain JSON — you can open them in any text editor, load them into pandas for analysis, or feed them into a visualization tool. This makes it easy to build custom debugging and monitoring workflows around your agents.

## Using Traces to Analyze Fallback Behavior

With a trace from an amphibious agent, you can see exactly when and why the framework switched from workflow to agent mode. This is invaluable for understanding whether your workflow steps are robust enough, or whether the agent is being called too often.

In [ ]:
# Analyze the trace for workflow-to-agent transitions
for i, phase in enumerate(trace.get("phases", [])):
    phase_type = phase.get("phase_type", "")
    goal = phase.get("goal", "")
    steps = phase.get("steps", [])

    print(f"Phase {i+1} [{phase_type}]: {goal}")
    for s in steps:
        output_type = s.get("output_type", "unknown")
        content = s.get("step_content", "")[:60]
        tools_used = [tc.get("tool_name") for tc in s.get("tool_calls", [])]
        print(f"  [{output_type}] {content}... → tools: {tools_used}")

This analysis view shows you the flow of control through the agent. You can see:

- Which phases ran in which order.
- The type of each step (`output_type` tells you whether it was a workflow step or an agent reasoning step).
- Which tools were called at each step.

If you notice the agent is frequently falling back on the same workflow step, that is a signal to either fix the workflow step or add error handling. If the agent is taking many steps to resolve a simple issue, you might need to improve the `CognitiveWorker` prompt.

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we learned how to see inside the agent's decision-making process:

- **Enable tracing** with `arun(..., trace_running=True)` to record every observation, decision, and tool call.
- **Access the trace** via `agent._agent_trace.build()` for a structured dictionary, or **save/load** with `.save("path.json")` and `AgentTrace.load("path.json")`.
- The trace is organized by **phases** (from Phase Annotations) and **steps** (individual OTC cycles), making it easy to navigate complex execution flows.
- Tracing is especially valuable for **amphibious agents**: you can see exactly when workflow steps failed, when the agent took over, and how it resolved the issue.
- Use traces for **debugging** (why did the agent make that decision?), **optimization** (which steps were redundant?), and **auditing** (what actions did the agent take?).

Execution tracing turns agent behavior from opaque to transparent, making it practical to build, debug, and trust complex agentic systems.